# Model 5: Naive Bayes
## IoT Saldırı Tipi Sınıflandırması (Multi-class)

Bu notebook, **Edge-IIoTset** veri seti üzerinde **Attack_type** kolonunu hedef değişken olarak kullanan
Naive Bayes modelini adım adım eğitir ve değerlendirir.

### Neden Naive Bayes?
- **Çok hızlı eğitim**: Kapalı formda çözüm (iterasyon yok)
- **Probabilistik model**: Her sınıf için olasılık tahmini verir
- **Baseline değeri**: En basit olasılıksal sınıflandırıcı olarak alt sınır belirler
- **Düşük bellek**: Model boyutu küçüktür

### Önemli Notlar
- **MinMaxScaler gerekli**: NB negatif feature değerlerini kabul etmez → [0, 1] aralığına ölçeklenir
- **weightCol desteklenmiyor**: Spark MLlib NaiveBayes `weightCol` parametresi desteklemez, bu yüzden sınıf dengesizliği modelin doğal smoothing'i ile telafi edilir
- **Bağımsızlık varsayımı**: Feature'lar arası korelasyon yok kabul edilir (gerçekte ihlal edilse bile genellikle iyi çalışır)

### Pipeline Adımları
1. Gold Delta Lake → Veri yükleme
2. Feature vektörleme + StringIndexer
3. Stratified Train/Test split (%80/%20)
4. MinMaxScaler → [0, 1] normalizasyon
5. NaiveBayes (multinomial)
6. CrossValidator ile hiperparametre arama (smoothing, modelType)
7. Test seti değerlendirme
8. Model analizi + Feature discriminative power
9. MLflow loglama

## 1. Kütüphaneler ve Spark Session

In [ ]:
import sys
import time
import math
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from pyspark.ml import Pipeline
from pyspark.ml.classification import NaiveBayes
from pyspark.ml.feature import MinMaxScaler
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

sys.path.insert(0, "/opt/bitnami/spark")

from spark.spark_session import get_spark
from ml.utils import (
    run_ml_pipeline_multiclass,
    evaluate_model_multiclass,
    compute_confusion_matrix_multiclass,
    log_to_mlflow,
)

print("Kütüphaneler yüklendi.")

In [ ]:
spark = get_spark("Notebook-NaiveBayes-Multiclass")
print(f"Spark version: {spark.version}")
print(f"App name: {spark.sparkContext.appName}")

## 2. Veri Yükleme ve Feature Hazırlığı

In [ ]:
SAMPLE_SIZE = 30000  # Hızlı test için (None = tüm veri)

stage_start = time.perf_counter()
train_df, test_df, feature_cols, label_index_model = run_ml_pipeline_multiclass(
    spark,
    sample_size=SAMPLE_SIZE,
    split_log_stats=True,
)
label_names = list(label_index_model.labels)
num_classes = len(label_names)

print(f"\nPipeline süresi: {time.perf_counter() - stage_start:.2f}s")
print(f"Feature sayısı: {len(feature_cols)}")
print(f"Sınıf sayısı: {num_classes}")
print(f"Sınıflar: {label_names}")

In [ ]:
train_count = train_df.count()
test_count = test_df.count()
print(f"Train: {train_count:,} satır")
print(f"Test:  {test_count:,} satır")

print("\nTrain seti sınıf dağılımı:")
train_df.groupBy("label").count().orderBy("label").show()

## 3. Model Pipeline Oluşturma

Naive Bayes pipeline'ı iki aşamadan oluşur:

1. **MinMaxScaler**: Feature'ları [0, 1] aralığına normalize eder. NB negatif değer kabul etmez.
2. **NaiveBayes**: Multinomial model tipi ile çok sınıflı sınıflandırma.

### Hiperparametreler
| Parametre | Açıklama | Aranacak Değerler |
|-----------|----------|------------------|
| `smoothing` | Laplace smoothing parametresi | 0.5, 1.0, 2.0 |
| `modelType` | NB model tipi | multinomial |

In [ ]:
scaler = MinMaxScaler(
    inputCol="features", outputCol="scaled_features",
    min=0.0, max=1.0,
)

nb = NaiveBayes(
    featuresCol="scaled_features",
    labelCol="label",
    modelType="multinomial",
    smoothing=1.0,
)

pipeline = Pipeline(stages=[scaler, nb])
print("Pipeline oluşturuldu: MinMaxScaler → NaiveBayes (multinomial)")
print("Not: NaiveBayes weightCol desteklemez — classWeight kullanılmaz.")

## 4. Cross Validation

In [ ]:
smoothing_values = [0.5, 1.0, 2.0]
model_type_values = ["multinomial"]
num_folds = 3

param_grid = (
    ParamGridBuilder()
    .addGrid(nb.smoothing, smoothing_values)
    .addGrid(nb.modelType, model_type_values)
    .build()
)

cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="f1",
    ),
    numFolds=num_folds,
    seed=42,
    parallelism=2,
)

total_cv_runs = len(param_grid) * num_folds
print(f"Grid boyutu: {len(param_grid)} kombinasyon")
print(f"Fold sayısı: {num_folds}")
print(f"Toplam fit: {total_cv_runs}")

## 5. Model Eğitimi

In [ ]:
print("Cross Validation başlatılıyor...")
cv_start = time.perf_counter()
cv_model = cv.fit(train_df)
cv_duration = time.perf_counter() - cv_start
print(f"CV tamamlandı! Süre: {cv_duration:.2f}s")

best_nb_model = cv_model.bestModel.stages[-1]
print(f"\nEn iyi parametreler:")
print(f"  modelType: {best_nb_model.getModelType()}")
print(f"  smoothing: {best_nb_model.getSmoothing()}")

In [ ]:
# CV sonuçları
avg_metrics = cv_model.avgMetrics
print("CV Sonuçları (F1-Score):")
print(f"{'#':<4} {'Parametreler':<35} {'Avg F1':>10}")
print("-" * 52)
cv_details = []
best_idx = int(max(range(len(avg_metrics)), key=lambda i: avg_metrics[i]))
for i, (params, score) in enumerate(zip(param_grid, avg_metrics)):
    ps = ", ".join(f"{p.name}={v}" for p, v in params.items())
    marker = " ← best" if i == best_idx else ""
    print(f"{i+1:<4} {ps:<35} {score:>10.4f}{marker}")
    cv_details.append({"index": i, "score": float(score), "params": ps})

cv_best_f1 = float(max(avg_metrics)) if avg_metrics else 0.0

## 6. Test Seti Değerlendirme

In [ ]:
predictions = cv_model.bestModel.transform(test_df)
metrics = evaluate_model_multiclass(predictions, num_classes=num_classes)

In [ ]:
confusion = compute_confusion_matrix_multiclass(predictions, label_names=label_names)

In [ ]:
print("\n" + "=" * 40)
print("SONUÇ ÖZETİ")
print("=" * 40)
print(f"Accuracy:  {metrics.get('accuracy', 0):.4f}")
print(f"F1-Score:  {metrics.get('f1_score', 0):.4f}")
print(f"Precision: {metrics.get('precision', 0):.4f}")
print(f"Recall:    {metrics.get('recall', 0):.4f}")

## 7. Naive Bayes Model Analizi

NB modeli iki temel bileşenden oluşur:
- **pi (class priors)**: log P(class) — her sınıfın önsel olasılığı
- **theta (feature log-likelihoods)**: log P(feature | class) — her feature'ın sınıf koşullu olasılığı

In [ ]:
# Model bileşenleri
pi = best_nb_model.pi.toArray().tolist()
theta = best_nb_model.theta.toArray()

print(f"Model Tipi:     {best_nb_model.getModelType()}")
print(f"Smoothing:      {best_nb_model.getSmoothing()}")
print(f"Sınıf Sayısı:   {len(pi)}")
print(f"Feature Sayısı: {theta.shape[1]}")

# Sınıf priori olasılıkları
print(f"\nSınıf Önsel Olasılıkları (Prior):")
print(f"{'Sınıf':<25} {'log P(c)':>10} {'P(c)':>10}")
print("-" * 48)
for i, log_p in enumerate(pi):
    cls_name = label_names[i] if i < len(label_names) else f"class_{i}"
    print(f"{cls_name:<25} {log_p:>10.4f} {math.exp(log_p):>10.4f}")

## 8. Feature Discriminative Power

NB'de doğrudan `featureImportances` yoktur. Bunun yerine **theta matrix**'inden
her feature için sınıflar arası log-likelihood **varyansı** hesaplanır:
- Yüksek varyans → Feature sınıfları güçlü ayırıyor
- Düşük varyans → Feature sınıflar arasında benzer dağılım gösteriyor

In [ ]:
# Discriminative power: sınıflar arası varyans
n_features = theta.shape[1]
scores = []
for j in range(n_features):
    col_vals = theta[:, j]
    mean_val = np.mean(col_vals)
    var_val = np.var(col_vals)
    scores.append((feature_cols[j], float(var_val)))

scores.sort(key=lambda x: x[1], reverse=True)
top_features = scores[:15]

print("Top 15 Discriminative Feature (log-likelihood varyansı):")
print(f"{'#':<4} {'Feature':<35} {'Disc. Power':>12}")
print("-" * 55)
for idx, (fname, power) in enumerate(top_features, start=1):
    print(f"{idx:<4} {fname:<35} {power:>12.6f}")

In [ ]:
# Discriminative Power Görselleştirmesi
top10 = scores[:10]
names = [f[0] for f in reversed(top10)]
values = [f[1] for f in reversed(top10)]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(names, values, color="#9C27B0", edgecolor="#6A1B9A", height=0.6)
for bar, val in zip(bars, values):
    ax.text(bar.get_width() + max(values) * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}", va="center", fontsize=9, fontweight="bold")

ax.set_xlabel("Discriminative Power (var of log-likelihoods)", fontsize=11)
ax.set_title("Naive Bayes — Top 10 Discriminative Feature (Multi-class)",
             fontsize=13, fontweight="bold")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

## 9. Sınıf Bazında Feature Analizi

In [ ]:
# Her sınıf için en yüksek log-likelihood'a sahip 5 feature
for i in range(min(num_classes, 5)):  # İlk 5 sınıf
    cls_name = label_names[i] if i < len(label_names) else f"class_{i}"
    pairs = sorted(zip(feature_cols, theta[i].tolist()), key=lambda x: x[1], reverse=True)[:5]
    print(f"\nSınıf '{cls_name}' — En yüksek log P(feature|class):")
    for j, (fname, val) in enumerate(pairs, 1):
        print(f"  {j}. {fname:<30} log_likelihood={val:.6f}")

## 10. MLflow'a Loglama

In [ ]:
best_params = {
    "modelType": str(best_nb_model.getModelType()),
    "smoothing": float(best_nb_model.getSmoothing()),
    "used_minmax_scaler": True,
    "numClasses": int(num_classes),
    "numFolds": num_folds,
    "grid_size": len(param_grid),
    "cv_total_fits": total_cv_runs,
    "label_column": "Attack_type",
}

metrics["cv_best_f1"] = cv_best_f1

confusion_metrics = {}
for i, name in enumerate(label_names):
    confusion_metrics[f"row_total_class_{i}"] = confusion["row_totals"][i]
    confusion_metrics[f"per_class_acc_{i}"] = confusion["per_class_acc"][i]

run_id = log_to_mlflow(
    run_name="naive_bayes_notebook",
    model_type="NaiveBayes",
    params=best_params,
    metrics={**metrics, **confusion_metrics},
    model=cv_model.bestModel,
    feature_importance=top_features[:10],
    tags={
        "source": "notebook",
        "model_index": "5",
        "classification_type": "multiclass",
        "baseline_model": "true",
    },
)
print(f"\nMLflow Run ID: {run_id}")

## Özet

Bu notebook'ta **Naive Bayes** modelini başarıyla eğittik:

- **MinMaxScaler** ile feature'lar [0, 1] aralığına normalize edildi
- **CrossValidator** ile smoothing parametresi optimize edildi
- Sınıf dengesizliği NB'nin doğal smoothing'i ile telafi edildi (weightCol desteklenmiyor)
- **theta matrix** analizi ile sınıf bazında feature davranışı incelendi
- **Discriminative power** (log-likelihood varyansı) ile feature önem sıralaması yapıldı
- Sonuçlar **MLflow**'a loglandı

Naive Bayes, en hızlı eğitilen model olup probabilistik bir baseline sağlar. Bağımsızlık varsayımı güçlü olsa da, pratikte genellikle iyi sonuçlar verir.

In [ ]:
spark.stop()
print("Spark oturumu kapatıldı.")